# Module 4 - Week 6, Topic 6: Cross-Validation & Hyperparameter Tuning
## Live Demo Notebook

**Scenario:** A Nigerian microfinance company wants the best possible loan default classifier.
We show:
1. Why a single split is unreliable
2. How k-Fold CV gives stable estimates
3. How GridSearchCV finds the optimal hyperparameters
4. How to avoid data leakage using Pipelines

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, recall_score

sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('loan_defaults_large.csv')
print('Shape:', df.shape)
print(f'Default rate: {df["defaulted"].mean():.1%}')

FEATURES = ['income', 'debt_ratio', 'num_late_payments', 'credit_score', 'employment_years']
TARGET   = 'defaulted'

X = df[FEATURES]
y = df[TARGET]

---
## Part 1 - The Problem with a Single Split

In [ ]:
# Same model, different random seeds - how much does accuracy vary?
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)

seeds = [0, 10, 42, 99, 200, 333, 500]
accuracies = []

print('Accuracy with a single train/test split - different random seeds:')
print()
for seed in seeds:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    model_s = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
    model_s.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, model_s.predict(X_te))
    accuracies.append(acc)
    print(f'  seed={seed:>3}: accuracy={acc:.2%}')

print()
print(f'Range: {min(accuracies):.2%} to {max(accuracies):.2%}')
print(f'Spread: {(max(accuracies)-min(accuracies))*100:.1f} percentage points')
print()
print('This variation is from random chance in the split - not model quality.')
print('Which accuracy would you report? They\'re all from the same model!')

---
## Part 2 - k-Fold Cross-Validation

In [ ]:
# 5-fold cross-validation - stable estimate regardless of split
model_cv = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)

cv_scores = cross_val_score(model_cv, X, y, cv=5, scoring='accuracy', n_jobs=-1)

print('5-Fold Cross-Validation Results:')
for i, score in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {score:.2%}')
print()
print(f'Mean accuracy: {cv_scores.mean():.2%}')
print(f'Std deviation: ±{cv_scores.std():.2%}')
print(f'Result: {cv_scores.mean():.2%} ± {cv_scores.std():.2%}')
print()
print('Compare: single-split spread was', f'{(max(accuracies)-min(accuracies))*100:.1f}%')
print('CV std is:', f'{cv_scores.std()*100:.1f}% - much more stable estimate.')

In [ ]:
# Use F1 as scoring (better for imbalanced data than accuracy)
cv_f1 = cross_val_score(model_cv, X, y, cv=5, scoring='f1', n_jobs=-1)

print('5-Fold CV with F1 scoring:')
for i, score in enumerate(cv_f1, 1):
    print(f'  Fold {i}: {score:.3f}')
print(f'\nMean F1: {cv_f1.mean():.3f} ± {cv_f1.std():.3f}')
print()
print('F1 scoring reveals more about model quality when classes are not balanced.')

---
## Part 3 - GridSearchCV: Find the Best Parameters

In [ ]:
# FIRST: hold out a test set before any tuning
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training (for CV + tuning): {len(X_train)} rows')
print(f'Test (locked until end):    {len(X_test)} rows')
print()
print('The test set is now locked. We will not look at it until the very end.')

In [ ]:
# Build a Pipeline (scaler + model)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=1000, random_state=42)),
])

# Parameter grid - use 'model__' prefix to target parameters inside the Pipeline
param_grid = {
    'model__C':       [0.01, 0.1, 1, 10, 100],
    'model__penalty': ['l1', 'l2'],
    'model__solver':  ['saga'],
}
# Total: 5 × 2 × 1 = 10 combinations × 5 folds = 50 fits

print(f'Grid size: {5*2} combinations × 5 folds = {5*2*5} model fits')
print()

gs = GridSearchCV(
    pipe, param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0,
)
gs.fit(X_train, y_train)

print('GridSearchCV complete.')
print(f'Best parameters: {gs.best_params_}')
print(f'Best CV F1:      {gs.best_score_:.3f}')

In [ ]:
# View all results
results_df = pd.DataFrame(gs.cv_results_)[[
    'params', 'mean_test_score', 'std_test_score', 'rank_test_score'
]].sort_values('rank_test_score')

print('Top 5 parameter combinations:')
print(results_df.head(5).to_string(index=False))

print()
print('Reading the table:')
print('  mean_test_score: average CV F1 across 5 folds')
print('  std_test_score:  variability - lower = more stable')
print('  rank: 1 = best combination')

In [ ]:
# FINAL EVALUATION on the held-out test set - one time only
best_model = gs.best_estimator_
y_pred     = best_model.predict(X_test)

print('=== Final Evaluation on Held-Out Test Set ===')
print(f'Best parameters: {gs.best_params_}')
print()
print(f'CV F1 (during tuning): {gs.best_score_:.3f}')
print(f'Test F1 (final):       {f1_score(y_test, y_pred):.3f}')
print()
gap = abs(gs.best_score_ - f1_score(y_test, y_pred))
print(f'Gap: {gap:.3f} (close = good; large gap = something is wrong)')

---
## Part 4 - RandomizedSearchCV: Faster Alternative

In [ ]:
from scipy.stats import randint

# Random Forest - many parameters, use RandomizedSearchCV
rf_pipe = Pipeline([
    ('model', RandomForestClassifier(random_state=42, n_jobs=-1))
])

param_dist = {
    'model__n_estimators':    randint(50, 300),
    'model__max_depth':       randint(3, 20),
    'model__min_samples_leaf': randint(1, 10),
}

rs = RandomizedSearchCV(
    rf_pipe, param_dist,
    n_iter=15,           # try 15 random combinations
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1,
    verbose=0,
)
rs.fit(X_train, y_train)

print('RandomizedSearchCV complete.')
print(f'Best parameters: {rs.best_params_}')
print(f'Best CV F1:      {rs.best_score_:.3f}')
print()
rs_pred = rs.best_estimator_.predict(X_test)
print(f'Test F1: {f1_score(y_test, rs_pred):.3f}')

---
## Part 5 - Data Leakage: Wrong vs Right

In [ ]:
# WRONG: scale before GridSearchCV - scaler sees validation fold statistics
scaler_wrong = StandardScaler()
X_train_scaled_wrong = scaler_wrong.fit_transform(X_train)  # sees ALL training rows

gs_wrong = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    {'C': [0.1, 1, 10], 'penalty': ['l2'], 'solver': ['lbfgs']},
    cv=5, scoring='f1', n_jobs=-1
)
gs_wrong.fit(X_train_scaled_wrong, y_train)

# CORRECT: scaler inside the Pipeline - fitted only on each training fold
pipe_correct = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=1000, random_state=42)),
])
gs_correct = GridSearchCV(
    pipe_correct,
    {'model__C': [0.1, 1, 10], 'model__penalty': ['l2'], 'model__solver': ['lbfgs']},
    cv=5, scoring='f1', n_jobs=-1
)
gs_correct.fit(X_train, y_train)

print('Data Leakage Comparison:')
print(f'  WRONG  (scaler before CV): CV F1 = {gs_wrong.best_score_:.3f}')
print(f'  CORRECT (scaler in Pipeline): CV F1 = {gs_correct.best_score_:.3f}')
print()

# Test both on the real held-out test set
X_test_scaled  = scaler_wrong.transform(X_test)
wrong_test_f1  = f1_score(y_test, gs_wrong.best_estimator_.predict(X_test_scaled))
correct_test_f1 = f1_score(y_test, gs_correct.best_estimator_.predict(X_test))

print(f'  WRONG  test F1:   {wrong_test_f1:.3f}')
print(f'  CORRECT test F1:  {correct_test_f1:.3f}')
print()
print('Leakage can make CV score appear artificially optimistic.')
print('The Pipeline prevents leakage automatically.')

---
## Summary

| Concept | Key code | Key insight |
|---|---|---|
| Single-split instability | `train_test_split(random_state=seed)` | Accuracy varies 7%+ from chance |
| k-Fold CV | `cross_val_score(model, X, y, cv=5)` | Mean ± std - stable, unbiased estimate |
| GridSearchCV | `GridSearchCV(pipe, param_grid, cv=5)` | Exhaustive search, 50–200 fits |
| RandomizedSearchCV | `RandomizedSearchCV(pipe, dist, n_iter=15)` | Faster, near-optimal for large grids |
| No leakage | `Pipeline([scaler, model])` inside `GridSearchCV` | Scaler fitted per fold, not globally |
| Final evaluation | `gs.best_estimator_.predict(X_test)` | Use test set once, after all tuning |

**Week 6 complete.** You can now build, evaluate, and tune any supervised ML model.

**Next - Module Demo:** All four algorithms applied to Access Bank loan data in one comprehensive session.